[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview — architecture → backward → sanity check

작은 tensor와 얕은 network를 쓰되 **원 모델의 핵심 계산 구조를 다른 toy 구조로 바꾸지 않는다.**

- GPT: causal self-attention
- ViT: patch tokens + CLS
- DiT: adaLN-Zero + flow velocity
- π0-style VLA: separate base/action-expert parameters + asymmetric joint attention


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 0. 공통 Transformer block

Q/K/V와 attention score를 직접 만든다. GPT에서는 causal mask를, ViT에서는 full attention을 사용한다.


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x, mask=None):
        batch, length, dim = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(
                ~mask[None, None],
                torch.finfo(scores.dtype).min,
            )

        weights = scores.softmax(dim=-1)
        hidden = (weights @ v).transpose(1, 2).contiguous().view(batch, length, dim)
        return self.out(hidden), weights


class TransformerBlock(nn.Module):
    def __init__(self, dim=24, heads=3):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x, mask=None):
        attn, weights = self.attn(self.norm1(x), mask)
        x = x + attn
        x = x + self.mlp(self.norm2(x))
        return x, weights


## A. Tiny GPT

Learned token/position embedding → causal Transformer → final norm → tied LM head를 유지한다.


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab=32, max_length=16, dim=24, depth=2):
        super().__init__()
        self.token = nn.Embedding(vocab, dim)
        self.position = nn.Embedding(max_length, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, 3) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, tokens, return_attention=False):
        length = tokens.size(1)
        positions = torch.arange(length, device=tokens.device)
        x = self.token(tokens) + self.position(positions)[None]
        mask = torch.tril(torch.ones(length, length, dtype=torch.bool, device=tokens.device))

        maps = []
        for block in self.blocks:
            x, weights = block(x, mask)
            maps.append(weights)

        logits = F.linear(self.norm(x), self.token.weight)
        return (logits, maps) if return_attention else logits


gpt = TinyGPT().to(device)
tokens = torch.tensor([[1, 2, 3, 4, 5, 6]], device=device)

logits = gpt(tokens[:, :-1])
gpt_loss = F.cross_entropy(logits.flatten(0, 1), tokens[:, 1:].flatten())
gpt_loss.backward()

_, maps = gpt(tokens, return_attention=True)
future_mass = maps[-1][0, 0].triu(diagonal=1).sum()

print("GPT loss:", gpt_loss.item())
print("future attention mass:", future_mass.item())


## B. Tiny ViT

Image → non-overlapping patch projection → CLS + learned position → full Transformer → classifier 구조를 유지한다.


In [ ]:
class TinyViT(nn.Module):
    def __init__(self, image_size=16, patch=4, dim=24, classes=10):
        super().__init__()
        self.patch = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        patch_count = (image_size // patch) ** 2

        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.randn(1, patch_count + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([TransformerBlock(dim, 3) for _ in range(2)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, image):
        x = self.patch(image).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(image.size(0), -1, -1), x], dim=1)
        x = x + self.position[:, : x.size(1)]

        for block in self.blocks:
            x, _ = block(x)

        return self.head(self.norm(x[:, 0]))


vit = TinyViT().to(device)
images = torch.randn(4, 3, 16, 16, device=device)
labels = torch.randint(0, 10, (4,), device=device)

vit_loss = F.cross_entropy(vit(images), labels)
vit_loss.backward()

print("ViT loss:", vit_loss.item())
print("patch-projection grad:", vit.patch.weight.grad.norm().item())


## C. Tiny DiT + Flow Matching

DiT block은 condition에서 attention/MLP 각각의 `shift, scale, gate`를 만드는 **adaLN-Zero** 구조를 사용한다.
학습 목표는 straight Flow Matching의 velocity `noise - data`다.


In [ ]:
def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(self, dim=24, heads=3):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = SelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)

        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )
        self.condition = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.condition[-1].weight)
        nn.init.zeros_(self.condition[-1].bias)

    def forward(self, x, condition):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.condition(condition).chunk(6, dim=-1)

        attn_input = modulate(self.norm1(x), shift_a, scale_a)
        attn, _ = self.attn(attn_input)
        x = x + gate_a[:, None] * attn

        mlp_input = modulate(self.norm2(x), shift_m, scale_m)
        return x + gate_m[:, None] * self.mlp(mlp_input)


class TinyDiT(nn.Module):
    def __init__(self, image_size=8, patch=2, dim=24):
        super().__init__()
        self.patch_size = patch
        self.patch = nn.Conv2d(1, dim, patch, stride=patch)
        token_count = (image_size // patch) ** 2
        self.position = nn.Parameter(torch.randn(1, token_count, dim) * 0.02)

        self.time = nn.Sequential(nn.Linear(1, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList([DiTBlock(dim, 3) for _ in range(2)])
        self.out = nn.Linear(dim, patch * patch)

    def forward(self, image, t):
        x = self.patch(image).flatten(2).transpose(1, 2) + self.position
        condition = self.time(t[:, None])

        for block in self.blocks:
            x = block(x, condition)

        patches = self.out(x).transpose(1, 2)
        return F.fold(
            patches,
            output_size=(image.size(-2), image.size(-1)),
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )


dit = TinyDiT().to(device)
data = torch.randn(4, 1, 8, 8, device=device)
noise = torch.randn_like(data)
t = torch.rand(4, device=device)

z_t = (1 - t[:, None, None, None]) * data + t[:, None, None, None] * noise
velocity_target = noise - data

velocity = dit(z_t, t)
dit_loss = F.mse_loss(velocity, velocity_target)
dit_loss.backward()

print("DiT velocity:", velocity.shape)
print("condition grad:", dit.blocks[0].condition[-1].weight.grad.norm().item())


## D. Tiny π0-style VLA flow policy

π0-like section도 generic Transformer 하나로 합치지 않는다.

- vision/language prefix: base/VLM parameter set
- state/noisy actions: separate action-expert parameter set
- Q/K/V만 합쳐 joint attention
- prefix → suffix attention은 금지, suffix → prefix는 허용
- action expert가 flow velocity를 직접 예측하고 backward된다


In [ ]:
def pi0_mask(prefix_len, suffix_len, device):
    mask = torch.zeros(
        prefix_len + suffix_len,
        prefix_len + suffix_len,
        dtype=torch.bool,
        device=device,
    )
    mask[:prefix_len, :prefix_len] = True
    mask[prefix_len:, :] = True
    return mask


class SplitJointBlock(nn.Module):
    def __init__(self, dim=24, heads=3):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads

        self.base_norm = nn.RMSNorm(dim)
        self.base_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.base_out = nn.Linear(dim, dim, bias=False)
        self.base_mlp = nn.Sequential(
            nn.RMSNorm(dim), nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim)
        )

        self.expert_norm = nn.RMSNorm(dim)
        self.expert_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.expert_out = nn.Linear(dim, dim, bias=False)
        self.expert_mlp = nn.Sequential(
            nn.RMSNorm(dim), nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim)
        )

    def qkv(self, x, projection):
        batch, length, dim = x.shape
        qkv = projection(x).view(batch, length, 3, self.heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)

    def forward(self, prefix, suffix):
        bq, bk, bv = self.qkv(self.base_norm(prefix), self.base_qkv)
        eq, ek, ev = self.qkv(self.expert_norm(suffix), self.expert_qkv)

        q = torch.cat([bq, eq], dim=2)
        k = torch.cat([bk, ek], dim=2)
        v = torch.cat([bv, ev], dim=2)

        mask = pi0_mask(prefix.size(1), suffix.size(1), prefix.device)
        attended = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

        p_len = prefix.size(1)
        base = attended[:, :, :p_len].transpose(1, 2).contiguous().flatten(2)
        expert = attended[:, :, p_len:].transpose(1, 2).contiguous().flatten(2)

        prefix = prefix + self.base_out(base)
        suffix = suffix + self.expert_out(expert)
        return prefix + self.base_mlp(prefix), suffix + self.expert_mlp(suffix), mask


class TinyPi0(nn.Module):
    def __init__(self, dim=24, horizon=5, action_dim=3):
        super().__init__()
        self.horizon = horizon
        self.vision = nn.Linear(10, dim)
        self.language = nn.Embedding(32, dim)

        self.state = nn.Linear(6, dim)
        self.action = nn.Linear(action_dim, dim)
        self.time = nn.Linear(1, dim)

        self.block = SplitJointBlock(dim, 3)
        self.velocity = nn.Linear(dim, action_dim)

    def forward(self, vision, language, state, noisy_actions, t):
        prefix = torch.cat([self.vision(vision), self.language(language)], dim=1)

        state_token = self.state(state).unsqueeze(1)
        time_token = self.time(t[:, None]).unsqueeze(1)
        action_tokens = self.action(noisy_actions)
        suffix = torch.cat([state_token, time_token, action_tokens], dim=1)

        prefix, suffix, mask = self.block(prefix, suffix)
        return self.velocity(suffix[:, -self.horizon:]), mask


pi0 = TinyPi0().to(device)
vision = torch.randn(3, 3, 10, device=device)
language = torch.randint(0, 32, (3, 4), device=device)
state = torch.randn(3, 6, device=device)

actions = torch.randn(3, 5, 3, device=device)
noise = torch.randn_like(actions)
t = torch.rand(3, device=device)

x_t = t[:, None, None] * noise + (1 - t[:, None, None]) * actions
target_velocity = noise - actions

pred_velocity, mask = pi0(vision, language, state, x_t, t)
pi0_loss = F.mse_loss(pred_velocity, target_velocity)
pi0_loss.backward()

print("π0 velocity:", pred_velocity.shape)
print("prefix -> action:", bool(mask[0, -1]))
print("action -> prefix:", bool(mask[-1, 0]))
print("base QKV grad:", pi0.block.base_qkv.weight.grad.norm().item())
print("expert QKV grad:", pi0.block.expert_qkv.weight.grad.norm().item())


## References and provenance

- **GPT / Transformer** — causal self-attention, pre-norm residual, tied LM head.
- **ViT** — patch projection, CLS token, learned position, full self-attention.
- **DiT** — adaLN-Zero conditioned Transformer + Flow Matching velocity regression.
- **π0 / openpi** — base prefix와 별도 action-expert parameter set, asymmetric joint attention, noisy action-horizon flow matching.

대규모 pretrained backbone은 tiny projection으로 줄였지만, 각 모델을 정의하는 핵심 계산 그래프는 유지한다.
